# Fraudulent transaction detection

This notebook is a lightweight companion to the reproducible training pipeline. The reusable implementation lives in `src/fraud_detection`; run `fraud-train` to regenerate all published results.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from fraud_detection.data import TARGET, load_transactions

## Load the data

The loader selects only model-relevant columns and assigns compact data types. Account identifiers are deliberately excluded from the modeling workflow.

In [ ]:
data_path = Path("../data/Fraud.csv")
transactions = load_transactions(data_path)
transactions.info()

In [ ]:
summary = pd.Series(
    {
        "transactions": len(transactions),
        "fraudulent_transactions": int(transactions[TARGET].sum()),
        "fraud_rate": transactions[TARGET].mean(),
        "time_steps": transactions["step"].nunique(),
    }
)
summary

## Class imbalance

Fraud is extremely rare, so accuracy is not an informative primary metric. The project emphasizes average precision, precision, recall, F2, and the confusion matrix.

In [ ]:
counts = transactions[TARGET].value_counts().sort_index()
ax = counts.plot.bar(color=["#4C78A8", "#E45756"], logy=True)
ax.set_xticklabels(["Legitimate", "Fraud"], rotation=0)
ax.set_ylabel("Transactions (log scale)")
ax.set_title("Target distribution")
plt.show()

## Transaction types

The next table separates transaction volume from fraud rate. This avoids confusing a common transaction type with a risky one.

In [ ]:
by_type = transactions.groupby("type", observed=True)[TARGET].agg(["count", "sum"])
by_type["fraud_rate"] = by_type["sum"] / by_type["count"]
by_type.sort_values("fraud_rate", ascending=False)

## Reproduce the model results

From the repository root, run:

```bash
uv run fraud-train
```

The command uses a temporal holdout, tunes the decision threshold only on validation data, and writes metrics, plots, a model card, feature importance, and a serialized model to `artifacts/`.